# Financial Dataset Generation for Symbolic Regression

This notebook processes financial data for S&P 500 and Apple stock to create datasets suitable for symbolic regression analysis. We aim to find latent equations that predict stock price movements based on various market indicators.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## Load and Explore Financial Data

In [ ]:
# Load the combined financial dataset
df = pd.read_csv('csvs/financial_combined.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f"Dataset shape: {df.shape}")
print(f"Symbols: {df['Symbol'].unique()}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Basic statistics
df.describe()

## Prepare Datasets for Symbolic Regression

We'll create multiple prediction tasks:
1. **Next-day return prediction**: Predict tomorrow's return based on today's features
2. **5-day return prediction**: Predict 5-day ahead returns
3. **Volatility prediction**: Predict future volatility
4. **Price direction**: Predict if price will go up or down

In [ ]:
def create_prediction_datasets(df):
    """
    Create multiple prediction datasets from financial data
    """
    datasets = {}
    
    for symbol in df['Symbol'].unique():
        symbol_df = df[df['Symbol'] == symbol].copy()
        symbol_df = symbol_df.sort_values('Date').reset_index(drop=True)
        
        # Feature columns (predictors)
        feature_cols = [
            'Open', 'High', 'Low', 'Close', 'Volume',
            'High_Low_Ratio', 'Volume_MA_5', 'Volume_MA_20',
            'Price_MA_5', 'Price_MA_20', 'Volatility_5', 'Volatility_20',
            'RSI', 'DayOfWeek', 'Month', 'Volume_Ratio'
        ]
        
        # Dataset 1: Next-day return prediction
        next_day_df = symbol_df[feature_cols].copy()
        next_day_df['target'] = symbol_df['Daily_Return'].shift(-1)  # Next day's return
        next_day_df = next_day_df.dropna()
        datasets[f'{symbol}_next_day_return'] = next_day_df
        
        # Dataset 2: 5-day return prediction
        five_day_df = symbol_df[feature_cols].copy()
        five_day_df['target'] = symbol_df['Price_Change_5d'].shift(-5)  # 5-day ahead return
        five_day_df = five_day_df.dropna()
        datasets[f'{symbol}_5day_return'] = five_day_df
        
        # Dataset 3: Volatility prediction
        vol_df = symbol_df[feature_cols].copy()
        vol_df['target'] = symbol_df['Volatility_5'].shift(-5)  # Future 5-day volatility
        vol_df = vol_df.dropna()
        datasets[f'{symbol}_volatility'] = vol_df
        
        # Dataset 4: Price direction (binary classification)
        direction_df = symbol_df[feature_cols].copy()
        direction_df['target'] = (symbol_df['Daily_Return'].shift(-1) > 0).astype(int)  # 1 if up, 0 if down
        direction_df = direction_df.dropna()
        datasets[f'{symbol}_direction'] = direction_df
    
    return datasets

# Create all datasets
prediction_datasets = create_prediction_datasets(df)

print("Created prediction datasets:")
for name, dataset in prediction_datasets.items():
    print(f"{name}: {dataset.shape} (rows x cols)")

## Save Datasets for Symbolic Regression

In [ ]:
# Save each dataset
for name, dataset in prediction_datasets.items():
    filepath = f'csvs/{name}_symbolic_regression.csv'
    dataset.to_csv(filepath, index=False)
    print(f"Saved {name} to {filepath}")

# Create a summary dataset with all prediction tasks
summary_data = []
for name, dataset in prediction_datasets.items():
    symbol, task = name.split('_', 1)
    summary_data.append({
        'dataset_name': name,
        'symbol': symbol,
        'task': task,
        'num_samples': len(dataset),
        'num_features': len(dataset.columns) - 1,  # exclude target
        'target_mean': dataset['target'].mean(),
        'target_std': dataset['target'].std(),
        'target_min': dataset['target'].min(),
        'target_max': dataset['target'].max()
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('csvs/financial_datasets_summary.csv', index=False)
print("\nDataset Summary:")
print(summary_df.to_string(index=False))

## Feature Correlation Analysis

In [ ]:
# Analyze feature correlations for one of the datasets
sample_dataset = prediction_datasets['SPY_next_day_return']

# Calculate correlation matrix
corr_matrix = sample_dataset.corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix (SPY Next-Day Return Prediction)')
plt.tight_layout()
plt.show()

# Show correlations with target
target_corr = corr_matrix['target'].abs().sort_values(ascending=False)
print("\nFeatures most correlated with target (absolute correlation):")
print(target_corr.head(10))

## Data Distribution Analysis

In [ ]:
# Plot target distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

tasks = ['next_day_return', '5day_return', 'volatility', 'direction']
for i, task in enumerate(tasks):
    spy_data = prediction_datasets[f'SPY_{task}']['target']
    aapl_data = prediction_datasets[f'AAPL_{task}']['target']
    
    axes[i].hist(spy_data, alpha=0.7, label='SPY', bins=30)
    axes[i].hist(aapl_data, alpha=0.7, label='AAPL', bins=30)
    axes[i].set_title(f'{task.replace("_", " ").title()} Distribution')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Prepare Data for LibraryAugmentedSymbolicRegression.jl

Create a format that can be easily used with the symbolic regression framework.

In [ ]:
# Create a master dataset file that combines all tasks
master_data = []

for name, dataset in prediction_datasets.items():
    symbol, task = name.split('_', 1)
    
    # Add metadata columns
    for idx, row in dataset.iterrows():
        record = {
            'dataset_name': name,
            'symbol': symbol,
            'task': task,
            'sample_id': idx
        }
        
        # Add all features and target
        for col in dataset.columns:
            record[col] = row[col]
        
        master_data.append(record)

master_df = pd.DataFrame(master_data)
master_df.to_csv('csvs/financial_master_dataset.csv', index=False)
print(f"Created master dataset with {len(master_df)} samples")
print(f"Columns: {list(master_df.columns)}")

## Summary

We have created multiple financial datasets for symbolic regression:
1. **Next-day return prediction**: Short-term price movement prediction
2. **5-day return prediction**: Medium-term price movement prediction  
3. **Volatility prediction**: Risk/uncertainty prediction
4. **Price direction**: Binary classification of price movements

Each dataset includes engineered features like moving averages, volatility measures, RSI, and volume ratios that could potentially form the basis of latent equations describing market behavior.

The datasets are now ready for symbolic regression analysis to discover mathematical relationships that govern stock price movements.